In [1]:
import os, json, gc
from collections import OrderedDict

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import scipy.io as sio

from tqdm.notebook import tqdm

from sklearn.model_selection import KFold, train_test_split, StratifiedKFold
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error

import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader, Subset
import torch.distributed as dist
import torch.multiprocessing as mp
from torch.nn.parallel import DistributedDataParallel as DDP

In [3]:
%load_ext autoreload
%autoreload 2

import sys
sys.path.append("..")
from src.preprocess import EmgEncoder

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [4]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device

device(type='cuda')

# データの前処理

In [5]:
num_emg_channels = 16
num_axis = 3
input_length = 1000
output_length = 30
split_width = input_length // output_length

train = sio.loadmat("../data/train.mat")
test = sio.loadmat("../data/test.mat")

X = []
y = []
test_X = []

# ユーザーごとに前処理を行う
users = ["0001", "0002", "0003", "0004"]
input_scalers = {}
output_scalers = {}
for user in users:
    train_x = train[user][0][0][0]
    train_y = train[user][0][0][1]
    test_x = test[user][0][0][0]
    
    # EMGのチャンネルごとに正規化
    input_scaler = {}
    for channel_id in range(num_emg_channels):
        _max = np.max(train_x[:, channel_id, :])
        _min = np.min(train_x[:, channel_id, :])
        train_x[:, channel_id, :] = (train_x[:, channel_id, :] - _min) / (_max - _min)
        test_x[:, channel_id, :] = (test_x[:, channel_id, :] - _min) / (_max - _min)
        input_scaler[channel_id] = (_min, _max)
    input_scalers[user] = input_scaler
    
    # 加速度の軸ごとに正規化
    output_scaler = {}
    for axis in range(num_axis):
        _max = np.max(train_y[:, axis, :])
        _min = np.min(train_y[:, axis, :])
        train_y[:, axis, :] = (train_y[:, axis, :] - _min) / (_max - _min)
        output_scaler[axis] = (_min, _max)
    output_scalers[user] = output_scaler
    
    # 入出力を分割して保存
    trial_x = []
    trial_y = []
    trial_test_x = []
    for _x, _y, _x_test in zip(train_x, train_y, test_x):
        _x = np.array_split(_x[:, :990], 30, axis=1)
        _y = np.split(_y, 30, axis=1)
        _x_test = np.array_split(_x_test[:, :990], 30, axis=1)
        
        trial_x += _x
        trial_y += _y
        trial_test_x += _x_test
    X += trial_x
    y += trial_y
    test_X += trial_test_x

X = np.array(X)
y = np.array(y)
test_X = np.array(test_X)

X.shape, y.shape, test_X.shape

((37770, 16, 33), (37770, 3, 1), (37770, 16, 33))

# データセットの作成

In [6]:
class CustomDataset(Dataset):
    def __init__(self, x, y=None, v_axis: int=0):
        self.x = x
        self.y = y
        self.v_axis = v_axis
    
    def __len__(self):
        return len(self.x)
    
    def __getitem__(self, idx):
        if self.y is None:
            return torch.tensor(self.x[idx].T, dtype=torch.float32)
        
        return torch.tensor(self.x[idx].T, dtype=torch.float32), torch.tensor(self.y[idx, self.v_axis, :], dtype=torch.float32)

dataset_many2one = CustomDataset(X, y)
test_dataset = CustomDataset(test_x)

sample_x, sample_y = next(iter(dataset_many2one))
sample_x.shape, sample_y.shape

(torch.Size([33, 16]), torch.Size([1]))

# モデルの定義

In [12]:
class CustomLSTM(nn.Module):
    def __init__(
        self,
        feature_size: int=16,
        len_sequence: int=33,
        hidden_size: int=32, 
        n_layers: int=1
    ) -> None:
        
        super().__init__()
        self.feature_size = feature_size
        self.hidden_size = hidden_size
        self.len_sequence = len_sequence
        self.n_layers = n_layers
        
        self.lstm1 = nn.LSTM(
            input_size=feature_size, 
            hidden_size=hidden_size, 
            num_layers=n_layers,
            batch_first=True
        )
        # self.flatten = nn.Flatten(start_dim=0)
        self.linear1 = nn.Linear(hidden_size, 1)
        
    def forward(self, x) -> torch.Tensor:
        output, (hn, cn) = self.lstm1(x)
        # output = self.flatten(output)
        print(output.size())
        print(output[-1].size())
        output = self.linear1(output[-1])
        return output

feature_size = 16
len_sequence = 33
hidden_size = 32
n_layers = 4
model = CustomLSTM(
    feature_size=feature_size,
    len_sequence=len_sequence,
    hidden_size=hidden_size,
    n_layers=n_layers
)
# dummy_input = torch.randn(1, len_sequence, feature_size)
sample_x, sample_y = next(iter(dataset_many2one))
print(model(sample_x))

# from torchinfo import summary
# print(summary(model, input_size=(1, 1000, 16)))

torch.Size([33, 32])
torch.Size([32])
tensor([-0.0453], grad_fn=<ViewBackward0>)


# モデルの学習